# EDA – metadata her ze Steamu

## Popis dat

#### Načtení dat
Datum stažení je čas poslední změny souboru `games.json`.

In [ ]:
import pandas as pd

from prepare_data import REPO, REVISION, download_metadata, load_metadata

path = download_metadata()
df = load_metadata(path)
download_date = pd.Timestamp(path.stat().st_mtime, unit="s").normalize()

print("Dataset:", REPO)
print("Revize:", REVISION)
print("Staženo:", download_date.date())

#### Počet řádků a sloupců

In [ ]:
df.shape

#### Náhled dat

In [ ]:
df.head()

#### Sloupce a datové typy

In [ ]:
df.info()

#### Pomocné sloupce
Počet recenzí a datum vydání se používají ve více buňkách, proto je spočítám jednou.

In [ ]:
n_reviews = df["positive"] + df["negative"]
release = pd.to_datetime(df["release_date"], format="mixed", errors="coerce")

## 1. Přehled recenzí

#### Celkový počet recenzí

In [ ]:
pos = df["positive"].sum()
neg = df["negative"].sum()
total = pos + neg
print("Celkem recenzí:", total)
print(f"Pozitivní: {pos} ({pos / total:.1%})")
print(f"Negativní: {neg} ({neg / total:.1%})")

#### Hry s nulovým počtem recenzí
Zjistím, kolik her má 0 recenzí. Hry s `recommendations > 0` recenze určitě mají,
nula u nich tedy neznamená „žádné recenze“, ale chybějící data.

In [ ]:
zero = n_reviews == 0
print("Hry s 0 recenzemi:", zero.sum())
print("  z toho recommendations > 0 (chybějící label):", (zero & (df["recommendations"] > 0)).sum())

## 2. Extrémní hodnoty

#### Nejdražší hry

In [ ]:
cols = ["name", "price", "positive", "negative", "required_age", "dlc_count"]
display(df.nlargest(5, "price")[cols])

#### Hry s nejvíce recenzemi

In [ ]:
top = df.loc[n_reviews.nlargest(5).index, cols].copy()
for c in ["positive", "negative"]:
    top[c] = [f"{v} ({v / n:.1%})" for v, n in zip(top[c], n_reviews[top.index])]
display(top)

#### Věková hranice a počet DLC

In [ ]:
print("Hodnoty required_age:", sorted(df["required_age"].unique()))
print("Maximální dlc_count:", df["dlc_count"].max())

## 3. Datum vydání

#### Kontrola data vydání

In [ ]:
print("Nepřevoditelná data:", release.isna().sum())
print("Rozsah data vydání:", release.min(), "-", release.max())
print("Vydáno po datu stažení:", (release > download_date).sum())

## 4. Duplicity
Jednotkou rozdělení dat je hra – stejná hra se nesmí objevit zároveň v trénovací a testovací množině.

#### Duplicitní app_id

In [ ]:
print("Duplicitní app_id:", df.index.duplicated().sum())

#### Stejná hra pod více app_id
Hledám hry se stejným názvem a stejnými vývojáři pod různými `app_id`.

In [ ]:
key = df["name"].str.lower() + " | " + df["developers"].map(", ".join)
dup = df[key.duplicated(keep=False)].assign(key=key, n_reviews=n_reviews)
print(f"{dup.shape[0]} her sdílí název a vývojáře s jinou hrou:")
display(dup.sort_values("key")[["name", "release_date", "price", "n_reviews"]].head(20))

## 5. Nulové recenze podle roku vydání

#### Podíl her s 0 recenzemi po letech
Pro každý rok vydání spočítám:
- `games` – počet her,
- `zero_share` – podíl her s 0 recenzemi,
- `missing_share` – podíl her s 0 recenzemi mezi hrami s `recommendations > 0`, tedy podíl chybějících labelů.

In [ ]:
year = release.dt.year
has_rec = df["recommendations"] > 0
table = pd.DataFrame({
    "games": zero.groupby(year).size(),
    "zero_share": zero.groupby(year).mean(),
    "missing_share": zero[has_rec].groupby(year[has_rec]).mean(),
}).round(3)
table.loc[2010:]